# Homework 6

Kai Rothe & Karim Zaghw

In [115]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mutual_info_score

## Exercise 1

Let random variable $X$ be a discrete random variable with values
{−2, −1, 0, 1, 2}, obtained with the same probability $p = \tfrac15$. Let
the second random variable $Y$ be $Y = (X+1)^2$.

(a) Compute analytically $cov(X, Y)$ (2 points)

Let $X$ be a discrete random variable with values $\{-2, -1, 0, 1, 2\}$, each with probability $p = \frac{1}{5}$. Define $Y = (X+1)^2$.

The covariance can be calculated as:
$$
\mathrm{cov}(X, Y) = \mathbb{E}[XY] - \mathbb{E}[X]\mathbb{E}[Y]
$$

First, compute $\mathbb{E}[X]$:
$$
\mathbb{E}[X] = \sum_{x} x \cdot p(x) = \frac{1}{5}(-2) + \frac{1}{5}(-1) + \frac{1}{5}(0) + \frac{1}{5}(1) + \frac{1}{5}(2) = 0
$$

Now, compute $\mathbb{E}[XY]$, using $p(y = (x + 1)^2 | x) = 1$, thus $p(y \neq (x + 1)^2 | x) = 0$:
$$
\mathbb{E}[XY] = \sum_{x, y} x \cdot y \cdot p(x, y) = \sum_{x, y} x \cdot y \cdot p(y | x) p(x) = \sum_{x} x \cdot (x+1)^2 \cdot p(x)
$$

\begin{align*}
x = -2: &\quad -2 \cdot 1 = -2 \\
x = -1: &\quad -1 \cdot 0 = 0 \\
x = 0: &\quad 0 \cdot 1 = 0 \\
x = 1: &\quad 1 \cdot 4 = 4 \\
x = 2: &\quad 2 \cdot 9 = 18 \\
\end{align*}

$$
\mathbb{E}[XY] = \frac{1}{5}(-2 + 0 + 0 + 4 + 18) = \frac{20}{5} = 4
$$

Therefore,
$$
\mathrm{cov}(X, Y) = \mathbb{E}[XY] - \mathbb{E}[X]\mathbb{E}[Y] = 4 - 0 = 4
$$

(b) Compute the mutual information between X and Y in bits (2 points).

The mutual information between $X$ and $Y$ is given by:
$$
M(X; Y) = H(X) - H(X|Y)
$$

Since $Y = (X+1)^2$ is a deterministic function of $X$, knowing $Y$ gives at most two possible values for $X$ (except for $Y=1$ which corresponds to $X=-2$ or $X=0$). Let's compute the probabilities:

Possible values:
- $X \in \{-2, -1, 0, 1, 2\}$, each with $p(x) = \frac{1}{5}$
- $Y = (X+1)^2 \implies Y \in \{0, 1, 4, 9\}$

Mapping:
- $X = -2 \implies Y = 1$
- $X = -1 \implies Y = 0$
- $X = 0 \implies Y = 1$
- $X = 1 \implies Y = 4$
- $X = 2 \implies Y = 9$

So,
- $p(Y=0) = \frac{1}{5}$
- $p(Y=1) = \frac{2}{5}$
- $p(Y=4) = \frac{1}{5}$
- $p(Y=9) = \frac{1}{5}$

$H(X)$:
$$
H(X) = -\sum_{x} p(x) \log_2 p(x) = -5 \cdot \frac{1}{5} \log_2 \frac{1}{5} = \log_2 5 \approx 2.322 \text{ bits}
$$

$H(X|Y)$:
- For $Y=0$: $X=-1$ (probability 1) $\implies H(X|Y=0) = 0$
- For $Y=1$: $X=-2$ or $X=0$, each with probability $\frac{1}{2}$:
    $$
    H(X|Y=1) = -2 \cdot \frac{1}{2} \log_2 \frac{1}{2} = 1
    $$
- For $Y=4$: $X=1$ (probability 1) $\implies H(X|Y=4) = 0$
- For $Y=9$: $X=2$ (probability 1) $\implies H(X|Y=9) = 0$

Average:
$$
H(X|Y) = p(Y=0) \cdot 0 + p(Y=1) \cdot 1 + p(Y=4) \cdot 0 + p(Y=9) \cdot 0 = \frac{2}{5} \cdot 1 = 0.4 \text{ bits}
$$

Therefore,
$$
M(X; Y) = H(X) - H(X|Y) = \log_2 5 - 0.4 \approx 2.322 - 0.4 = 1.922 \text{ bits}
$$

(c) Simulate 100, 1000, 10000 data realization of these processes and compute the covariances and mutual information based on the data. Compute the mutual information both with a library and yourself using the formula. (3 points)

In [ ]:
# Possible values for X
X_vals = np.array([-2, -1, 0, 1, 2])
p = 1/5

results = {}

for n in [100, 1000, 10000]:
    # Simulate X
    X_sim = np.random.choice(X_vals, size=n, p=[p]*len(X_vals))
    # Compute Y
    Y_sim = (X_sim + 1)**2

    # Covariance
    cov = np.cov(X_sim, Y_sim, ddof=0)[0, 1]

    # Mutual information using sklearn (in nats), convert to bits
    mi_lib = mutual_info_score(X_sim, Y_sim) / np.log(2)

    # Mutual information by formula
    # Estimate joint and marginal probabilities
    y_edges = [-0.5, 0.5, 1.5, 4.5, 9.5]
    x_edges = [-2.5, -1.5, -0.5, 0.5, 1.5, 2.5]
    counts, _, _ = np.histogram2d(X_sim, Y_sim, bins=[x_edges, y_edges], density=False)
    pxy = counts / n 
    
    px = np.sum(pxy, axis=1) 
    py = np.sum(pxy, axis=0) 
    
    mi_manual = 0
    for i in range(5):
        for j in range(4):
            if pxy[i, j] > 0:
                mi_manual += pxy[i, j] * (np.log2(pxy[i, j]) - np.log2(px[i]) - np.log2(py[j]))

    results[n] = {
        'covariance': cov,
        'mutual_info_library': mi_lib,
        'mutual_info_manual': mi_manual
    }

for n, res in results.items():
    print(f"n={n}: covariance={res['covariance']:.3f}, "
          f"mutual_info_library={res['mutual_info_library']:.3f}, "
          f"mutual_info_manual={res['mutual_info_manual']:.3f}")

n=100: covariance=4.527, mutual_info_library=1.850, mutual_info_manual=1.850
n=1000: covariance=3.980, mutual_info_library=1.923, mutual_info_manual=1.923
n=10000: covariance=3.957, mutual_info_library=1.921, mutual_info_manual=1.921


---
## Exercise 2
Suppose that we have a neuron which, in a given time period, will fire
with probability 0.2, yielding a Bernoulli distribution for the neuron’s firing (denoted by the random variable R = 0 or 1) with p(R = 1) = 0.2.

(a) Compute the entropy H(R) of this distribution (calculated in bits, i.e., using the base 2 logarithm)? (1 point)


The entropy $H(R)$ of a Bernoulli random variable $R$ with $p(R=1) = 0.1$ and $p(R=0) = 0.9$ is:

$$
H(R) = -[p(R=1)\log_2 p(R=1) + p(R=0)\log_2 p(R=0)]
$$

Plugging in the values:

$$
H(R) = -[0.2 \log_2 0.2 + 0.8 \log_2 0.8] \\
$$

In [143]:
H_R = -0.2 * np.log2(0.2) - 0.8 * np.log2(0.8)
print(f"So, the entropy H(R) = {H_R:.3f} bits.")

So, the entropy H(R) = 0.722 bits.


(b) Now lets add a stimulus to the picture. Suppose that we think this
neuron's activity is related to a light flashing in the eye. Let us say that
the light is flashing in a given time period with probability 0.2. Call this
stimulus random variable S. If there is a flash, the neuron will fire with
probability 1/2. If there is no flash, the neuron will fire with probability
1/8. Call the random variable describing whether the neuron fires or not
R. Compute the mutual information I(S : R)? (3 points)\
*Hint: First, confirm that H(R) is the same as above by computing p(R).*

The mutual information $I(S : R)$ can be calculated as:

$$
I(S : R) = H(R) - H(R|S)
$$

##### i) Confirm $H(R)$
Given the stimulus $S$, the (conditional) probabilities are:
- If $S=1$ (flash occurs $p(S=1) = 0.2$), $p(R=1|S=1) = \frac{1}{2}$ and $p(R=0|S=1) = \frac{1}{2}$
- If $S=0$ (no flash, $p(S=0) = 0.8$), $p(R=1|S=0) = \frac{1}{8}$ and $p(R=0|S=0) = \frac{7}{8}$

Therefore 
\begin{align*}
P(R = 1) &= P(R = 1 | S = 0) \cdot P(S = 0) + P(R = 1 | S = 1) \cdot P(S = 1) \\
&= \frac{1}{8} \cdot 0.8 + \frac{1}{2} \cdot 0.2 = 0.1 + 0.1 = 0.2
\end{align*}


So $P(R = 0) = 1 - P(R = 1) = 0.8$, and 

\begin{align*}
H(R) &= -[p(R=1)\log_2 p(R=1) + p(R=0)\log_2 p(R=0)] \\
&= -[0.2 \log_2 0.2 + 0.8 \log_2 0.8] \approx 0.722 \, \text{bits}
\end{align*}

This matches the previously computed value of $H(R)$.

##### ii) Compute $H(R|S)$

The entropy $H(R|S)$ is computed as the weighted average of the entropies for $S=1$ and $S=0$:
$$
H(R|S) = p(S=1) \cdot H(R|S=1) + p(S=0) \cdot H(R|S=0)
$$

Where:
$$
H(R|S=1) = -\left[\frac{1}{2} \log_2 \frac{1}{2} + \frac{1}{2} \log_2 \frac{1}{2}\right] = 1 \, \text{bit}
$$
$$
H(R|S=0) = -\left[\frac{1}{8} \log_2 \frac{1}{8} + \frac{7}{8} \log_2 \frac{7}{8}\right] \approx 0.54 \, \text{bits}
$$

Thus:
$$
H(R|S) = 0.2 \cdot 1 + 0.8 \cdot 0.54 \approx 0.63 \, \text{bits}
$$

##### Step 3: Compute $I(S : R)$
Finally, the mutual information is:
$$
I(S : R) = H(R) - H(R|S) = 0.72 - 0.63 \approx 0.09 \, \text{bits}
$$

In [144]:
H_R_given_no_flash = - (1/8 * np.log2(1/8) + 7/8 * np.log2(7/8))
print(f"The entropy H(R | no flash) = {H_R_given_no_flash:.3f} bits.")

H_R_given_S = 0.2 + 0.8 * H_R_given_no_flash
print(f"The entropy H(R | S) = {H_R_given_S:.2f} bits.")

M_RS = H_R - H_R_given_S
print(f"The mutual information M(R; S) = {M_RS:.3f} bits.")

The entropy H(R | no flash) = 0.544 bits.
The entropy H(R | S) = 0.63 bits.
The mutual information M(R; S) = 0.087 bits.
